In [ ]:
''' 
   LAN or WAN  | Option
# [LAN]  [WAN] | ssh >> minimal interaction (easy local comms) 
# [LAN]   WAN  | server --no-browser >> headless-gui (end-user friendly)
#  LAN   [WAN] | remote desktop >>  >> (end-user friendly) 
'''
# todo: ssh sequence 'response codes'
# todo: no server gui lan detection methods
# todo: remote desktop install and auth process 
def send_link(process):
    if cli_enabled:
        ssh(input("Enter username: "), input("Enter ip_address: "):
        continue
    else:
        if lan() == True:
            remote_browser('127.0.0.1')
            continue
        else:
            remote_browser(input("enter endpoint ip:"))
            continue
    return session_id

In [ ]:
import re
import time
import sys

# ── Constants ─────────────────────────────────────────────────────────────────
MAX_RETRIES    = 3
RETRY_DELAY    = 5      # seconds between retry attempts
RETRY_TIMEOUT  = 30     # seconds before abandoning a single attempt

# ── Helpers ───────────────────────────────────────────────────────────────────

def valid_ip(ip: str) -> bool:
    """Basic IPv4 validation."""
    pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
    if not re.match(pattern, ip):
        return False
    return all(0 <= int(o) <= 255 for o in ip.split('.'))

def get_input(prompt: str, validator=None) -> str:
    """Prompt until a valid non-empty value is entered."""
    while True:
        value = input(prompt).strip()
        if value and (validator is None or validator(value)):
            return value
        print("  [!] Invalid input — try again.")

def cli_enabled(process: dict) -> bool:
    """
    Resolve CLI preference from process config.
    Can be set at construction or toggled at runtime.
    Defaults to False (GUI) if not specified.
    """
    return process.get('cli_enabled', False)

# ── Main Function ─────────────────────────────────────────────────────────────

def send_link(process: dict):
    """
    Establish a connection to the endpoint.

    Connection matrix:
      CLI  + LAN/WAN → SSH              (minimal interaction, easy local comms)
      GUI  + LAN     → server --no-browser  (headless, end-user friendly)
      GUI  + WAN     → remote desktop       (end-user friendly)

    Extreme LAN case:
      New node installs software → generates credentials → transmits back.
      All subsequent connections route through node 0 via SSH/download tunnel.
      Applies to SIPR nets, air-gap-adjacent, or self-sustaining LAN nodes.

    cli_enabled is read from process dict — set at construction or toggled
    at runtime. New credential circumstances may override the stored preference.
    """

    session_id = None
    on_lan     = lan()
    use_cli    = cli_enabled(process)

    # ── Branch 1: CLI / SSH ───────────────────────────────────────────────────
    if use_cli:
        username   = get_input("Enter username: ")
        ip_address = get_input("Enter IP address: ", valid_ip)
        attempts   = 0

        while attempts < MAX_RETRIES:
            try:
                print(f"  [SSH] Attempt {attempts + 1} of {MAX_RETRIES}...")
                session_id = ssh(username, ip_address)

                if session_id is None:
                    raise ConnectionError("SSH returned no session.")

                # TODO: map SSH response codes once defined
                # 0   → success
                # 255 → connection refused / unreachable
                # paramiko: AuthenticationException, SSHException, socket.timeout
                break   # success — exit retry loop

            except ConnectionError as e:
                attempts += 1
                print(f"  [SSH] Failed: {e}")

                if attempts < MAX_RETRIES:
                    print(f"  [SSH] Retrying in {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print("  [SSH] Max retries reached — suspending session.")
                    suspend_session(process)
                    return None

    # ── Branch 2: GUI + LAN ───────────────────────────────────────────────────
    elif on_lan:
        attempts = 0

        while attempts < MAX_RETRIES:
            try:
                # TODO: LAN detection methods for no-browser server mode
                # Candidates: ARP scan, socket probe on known port,
                #             netmask check, gateway comparison
                session_id = remote_browser('127.0.0.1')

                if session_id is None:
                    raise ConnectionError("LAN remote browser returned no session.")

                # ── Extreme case ──────────────────────────────────────────────
                # New node provisions itself, generates credentials, transmits
                # back to node 0. Runs once — all future connections use the
                # established tunnel. Triggered by node capability detection.
                if is_extreme_case(process):
                    credentials = await_node_enrollment(process)

                    if credentials:
                        store_credentials(credentials, process)      # local first
                        remote_browser_enrollment(credentials, process)  # initiates
                                                                         # its own
                                                                         # transmittal
                                                                         # procedure
                    else:
                        raise ConnectionError("Node enrollment failed or timed out.")

                break   # success — exit retry loop

            except ConnectionError as e:
                attempts += 1
                print(f"  [LAN] Failed: {e}")

                if attempts < MAX_RETRIES:
                    print(f"  [LAN] Retrying in {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print("  [LAN] Max retries reached — suspending session.")
                    suspend_session(process)
                    return None

    # ── Branch 3: GUI + WAN ───────────────────────────────────────────────────
    else:
        ip_address = get_input("Enter endpoint IP: ", valid_ip)
        attempts   = 0

        while attempts < MAX_RETRIES:
            try:
                # TODO: remote desktop install + auth process
                # Recommended: Apache Guacamole (browser-based, no client install)
                # Auth flow:   generate one-time token → send via send_status()
                #              → validate on connect via receive_connection()
                session_id = remote_browser(ip_address)

                if session_id is None:
                    raise ConnectionError("WAN remote browser returned no session.")

                break   # success — exit retry loop

            except ConnectionError as e:
                attempts += 1
                print(f"  [WAN] Failed: {e}")

                if attempts < MAX_RETRIES:
                    print(f"  [WAN] Retrying in {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print("  [WAN] Max retries reached — suspending session.")
                    suspend_session(process)
                    return None

    return session_id